In [0]:
dbutils.widgets.text("catalog","fk_1392")
catalog = dbutils.widgets.get("catalog")
     
     

#gold.fact_order_line uses liquid clustering on order_date_key and store_id 

In [0]:
# TIME TRAVEL ON silver.products
# Version 3 = state after batch 1 (before batch 2 was loaded)
# Current = latest version of the table

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.time_travel_check AS
SELECT 
    COALESCE(a.product_id, b.product_id) AS product_id,
    CAST(a.selling_price AS DECIMAL(10,2)) AS selling_price_before_batch2,
    CAST(b.selling_price AS DECIMAL(10,2)) AS selling_price_current
FROM {catalog}.silver.products VERSION AS OF 3 a
FULL OUTER JOIN {catalog}.silver.products b
ON a.product_id = b.product_id
""")

In [0]:
#change data feed
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.product_changes_batch2 AS
SELECT 
    a.product_id,
    a._change_type,
    a.mrp AS mrp_after_batch2,
    a.selling_price,
    a.is_active ,
    a._commit_version
FROM table_changes('{catalog}.silver.products', 2, 3) a
LEFT ANTI JOIN {catalog}.ops.time_travel_check b
ON a.product_id = b.product_id
WHERE a._commit_version = (SELECT MAX(_commit_version) FROM table_changes('{catalog}.silver.products', 2, 3))""")

In [0]:
#3
spark.sql(f"OPTIMIZE {catalog}.gold.fact_order_line")

In [0]:
#4
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.ops.orders_drill DEEP CLONE {catalog}.silver.orders")
spark.sql(f"COMMENT ON TABLE {catalog}.ops.orders_drill IS 'Deep clone of silver.orders for the recovery drill'")




In [0]:
spark.sql(f"DELETE FROM {catalog}.ops.orders_drill WHERE store_id IN ('S01', 'S02')")


In [0]:

h = spark.sql(f"DESCRIBE HISTORY {catalog}.ops.orders_drill")
v_delete = h.where("operation = 'DELETE'").agg(F.max("version")).first()[0]
spark.sql(f"RESTORE TABLE {catalog}.ops.orders_drill TO VERSION AS OF {v_delete - 1}")


In [0]:
spark.sql(f"DESCRIBE HISTORY {catalog}.ops.orders_drill").select("version", "operation").show()
